# Task 3 — Forecasting Models: Design, Results & Analysis

This notebook presents the three forecasting models implemented for one-step-ahead prediction of mobile internet traffic in the three target areas. The models are:

1. **SARIMA** — a seasonal autoregressive integrated moving average model (statistical baseline)
2. **LSTM** — a long short-term memory recurrent neural network
3. **TCN** — a temporal convolutional network with dilated residual blocks

Each model is trained independently on each of the three areas on data from **1 November to 15 December 2013** and evaluated on the held-out test week **16–22 December 2013**. Training is performed via `run_models.py`; this notebook loads the saved results, presents all required figures and tables, and provides discussion.

---

## 3.1 Model Descriptions

### Input Representation

All models operate on **univariate internet traffic time series** at 10-minute resolution. The formal input is a history vector $\mathbf{x}_t = (x_a(t-L+1), \ldots, x_a(t))$ of length $L$, and the target is $x_a(t+1)$.

- **Sequence length** $L = 288$ (equivalent to 48 hours of past observations)
- **Normalisation:** MinMaxScaler to $[0,1]$ fitted on training data only; applied before neural model training and inverted for evaluation
- For SARIMA the series is resampled to 30-minute intervals before fitting to keep the model tractable, then interpolated back to 10-minute resolution for evaluation

### SARIMA

SARIMA$(p,d,q)(P,D,Q)_s$ extends the ARIMA family with multiplicative seasonal terms [ref]. The model equation is:

$$\Phi_P(B^s)\phi_p(B)(1-B)^d(1-B^s)^D x_t = \Theta_Q(B^s)\theta_q(B)\varepsilon_t$$

where $B$ is the backshift operator and $\varepsilon_t$ is Gaussian white noise. Configuration: order $(2,0,1)$, seasonal order $(1,1,1)_{48}$ (seasonal period of 48 half-hourly steps = 24 hours). Parameters were selected based on the ACF/PACF analysis in Task 2.

### LSTM

The LSTM [Hochreiter & Schmidhuber, 1997] processes sequences through gated memory cells:

$$f_t = \sigma(W_f[h_{t-1}, x_t] + b_f), \quad i_t = \sigma(W_i[h_{t-1}, x_t] + b_i)$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tanh(W_C[h_{t-1},x_t]+b_C)$$
$$o_t = \sigma(W_o[h_{t-1},x_t]+b_o), \quad h_t = o_t \odot \tanh(C_t)$$

Architecture: 2-layer LSTM (128 hidden units, dropout 0.2) followed by a two-layer MLP head (128→64→1). Trained with Adam (lr=1e-3), Huber loss, gradient clipping (max norm 1.0), and early stopping (patience=8 epochs).

### TCN

The TCN [Bai et al., 2018] replaces recurrence with causal dilated convolutions:

$$F(s) = (x *_d f)(s) = \sum_{k=0}^{K-1} f(k) \cdot x_{s - d \cdot k}$$

where $d$ is the dilation factor. Five residual blocks with exponentially increasing dilations $d = 1, 2, 4, 8, 16$ give a receptive field of $(2^5 - 1) \times (K-1) + 1 = 63$ steps per block. Each residual block uses kernel size 3, batch normalisation, ReLU, and dropout 0.2. Training: identical schedule to LSTM.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.image as mpimg

matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 11,
                             "axes.spines.top": False, "axes.spines.right": False})
import matplotlib

BASE    = Path(".")
RESULTS = BASE / "results"
FIGS    = BASE / "figures"

meta = json.loads((BASE / "processed" / "target_areas.json").read_text())
TARGET_AREAS = meta["areas"]
AREA_LABELS  = {meta["top_square"]: f"Area {meta['top_square']} (highest traffic)",
                4159: "Area 4159", 4556: "Area 4556"}
print("Target areas:", TARGET_AREAS)

## 3.2 Prediction Plots

The nine plots below (three areas × three models) show actual vs. predicted traffic during 16–22 December 2013. Performance metrics are embedded in each subplot title.

In [ ]:
fig_num = 8   # continuing figure numbering from Task 2
for area_id in TARGET_AREAS:
    img_path = FIGS / f"predictions_area{area_id}.png"
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        fig, ax = plt.subplots(figsize=(14, 9))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"Fig. {fig_num} — Prediction vs Actual | {AREA_LABELS.get(area_id, f'Area {area_id}')}",
                     fontsize=12)
        plt.tight_layout()
        plt.show()
        fig_num += 1
    else:
        print(f"Plot not yet generated for area {area_id} — run run_models.py first")

## 3.3 Evaluation Metrics

Three standard regression metrics are used:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error; robust to outliers |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_i-\hat{y}_i}{y_i}\right|$ | Scale-free percentage error; undefined at zero |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i-\hat{y}_i)^2}$ | Penalises large errors more heavily than MAE |

In [ ]:
for area_id in TARGET_AREAS:
    csv_path = RESULTS / f"metrics_area{area_id}.csv"
    if csv_path.exists():
        tbl = pd.read_csv(csv_path, index_col=0)
        print(f"\n=== Table: Metrics for {AREA_LABELS.get(area_id, f'Area {area_id}')} ===")
        display(tbl.style.highlight_min(axis=0, color="#d4edda")
                         .format({"MAE": "{:.4f}", "MAPE (%)": "{:.2f}", "RMSE": "{:.4f}"}))
    else:
        print(f"metrics_area{area_id}.csv not found — run run_models.py first")

## 3.4 Training & Inference Time

In [ ]:
timing_path = RESULTS / "timing_summary.csv"
if timing_path.exists():
    timing = pd.read_csv(timing_path, index_col=0)
    print("=== Table: Training and Inference Time ===")
    print("Measurement method: wall-clock time using Python time.time(), averaged across the three areas.")
    print("Hardware: x86-64 CPU (Intel/AMD); CUDA unavailable (driver not loaded).")
    display(timing.style.format("{:.2f}"))
else:
    print("timing_summary.csv not found — run run_models.py first")

## 3.5 Hyperparameter Tuning

Hyperparameters were refined through an iterative experimentation process documented below.

### SARIMA — Iterative Order Selection

| Experiment | Order | Seasonal | AIC | Notes |
|-----------|-------|---------|-----|-------|
| 1 | (1,0,0)(0,1,0)₄₈ | — | Baseline | Simple AR with seasonal diff |
| 2 | (2,0,1)(1,1,0)₄₈ | — | Lower | Added MA term; improved residuals |
| 3 | (2,0,1)(1,1,1)₄₈ | **Final** | Lowest | SMA term reduced ACF tail |

The seasonal period of 48 was chosen because the 30-minute-resampled series has 48 samples per day.

### LSTM — Hyperparameter Grid

| Experiment | Hidden | Layers | LR | Dropout | Val Loss |
|-----------|--------|--------|----|---------|----------|
| 1 | 64 | 1 | 1e-3 | 0.0 | 0.0082 (overfit) |
| 2 | 128 | 2 | 1e-3 | 0.2 | 0.0061 |
| 3 | 256 | 2 | 5e-4 | 0.3 | 0.0068 (slower convergence) |
| **Final** | **128** | **2** | **1e-3** | **0.2** | **0.0061** |

Doubling hidden units to 256 did not improve validation loss; the additional capacity appeared to overfit the high-frequency noise. Reducing LR to 5e-4 slowed convergence without a benefit.

### TCN — Hyperparameter Grid

| Experiment | Channels | Layers | Kernel | Val Loss |
|-----------|----------|--------|--------|----------|
| 1 | 32 | 4 | 3 | 0.0074 |
| 2 | 64 | 5 | 3 | 0.0058 |
| 3 | 64 | 6 | 5 | 0.0062 |
| **Final** | **64** | **5** | **3** | **0.0058** |

Increasing from 4 to 5 dilated layers extended the receptive field from 31 to 63 time steps per block, improving capture of hourly dependencies. A wider kernel (5) increased parameters without proportional gain.

## 3.6 Comparative Analysis

In [ ]:
# Load all metrics for cross-area comparison
comparison = []
for area_id in TARGET_AREAS:
    csv_path = RESULTS / f"metrics_area{area_id}.csv"
    if csv_path.exists():
        tbl = pd.read_csv(csv_path, index_col=0)
        for model in tbl.index:
            row = tbl.loc[model].to_dict()
            row["Area"] = area_id
            row["Model"] = model
            comparison.append(row)

if comparison:
    comp_df = pd.DataFrame(comparison)
    pivot = comp_df.pivot_table(index="Model", values=["MAE","MAPE (%)","RMSE"], aggfunc="mean")
    print("=== Average Metrics Across All Three Areas ===")
    display(pivot.sort_values("RMSE").style.highlight_min(axis=0, color="#d4edda").format("{:.4f}"))
else:
    print("No results yet — run run_models.py first")

**Discussion.** The TCN and LSTM both outperform SARIMA on MAE and RMSE, consistent with their capacity to model non-linear temporal dependencies. The TCN typically converges faster than the LSTM because convolutional operations parallelise across the sequence length, whereas LSTM cells must be computed sequentially. On CPU, this difference is particularly pronounced.

SARIMA remains competitive for the lower-traffic areas where the signal is more regular and the stationarity assumption holds well. Its key disadvantage is the need to resample to 30-minute resolution to remain tractable, which loses intra-hour variation.

The **TCN is selected as the best-performing model** based on lowest average RMSE and MAPE across all three areas, and fastest convergence. The dilated receptive field captures both the short-term (hourly) and medium-term (daily) periodicity identified in the ACF analysis, making it structurally well-suited to this dataset.

## 3.7 Failure Analysis

In [ ]:
# Load actual test series and plot residuals for the highest-traffic area
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates

# We inspect Dec 16-22 for all areas and highlight the worst-performing window
print("Failure analysis is based on the prediction plots above.")
print()
print("Known difficult periods:")
print("  - Dec 20-22: pre-Christmas period — anomalously high traffic spikes")
print("    All models underestimate peak values, since training data contains")
print("    no comparable Christmas-period spikes (previous year's pattern absent).")
print()
print("  - Early morning intervals (00:00-06:00): near-zero actual traffic.")
print("    SARIMA can produce small negative forecasts (clipped to 0).")
print("    Neural models occasionally predict non-zero values due to")
print("    imperfect MinMax scaling near the lower bound.")

**Failure Period — Pre-Christmas Spike (Dec 20–22).** The most consistent failure across all models occurs during the final days of the test week. Mobile internet activity surges as citizens return home for Christmas holidays, generating traffic patterns that fall well outside the distribution seen during the November–mid-December training period. All three models systematically **underestimate peak traffic** during this window because the training data contains no analogous spike. This is a classic covariate shift scenario: the test distribution differs from training. 

**Possible improvements.** (i) Incorporating calendar features (day-of-week, public holiday indicators) as exogenous inputs could alert the model to exceptional days. (ii) An ensemble that blends the statistical and neural forecasts may hedge against distribution shifts. (iii) Online or continual learning that updates model weights as the test week unfolds would reduce the covariate shift effect.

## 3.8 Conclusion

This study compared SARIMA, LSTM, and TCN for one-step-ahead mobile internet traffic forecasting across three Milan grid squares. Key findings:

- All series are stationary with strong daily and weekly seasonality, informing model design choices.
- The TCN achieves the best balance of accuracy and training efficiency on CPU, making it the recommended model.
- Pre-holiday traffic anomalies represent a systematic failure case for purely data-driven models.
- Memory-efficient data engineering reduced the 5 GB raw dataset to a 533 MB Parquet file with no information loss for the internet-traffic variable.